# Phase 1 EDA

Exploratory data analysis for the reputation intelligence assignment. This notebook intentionally performs profiling only. It does not clean, classify, score sentiment, or build dashboard logic.

In [ ]:
from __future__ import annotations

import logging
import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value: object) -> None:
        print(value)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from preprocessing import (
    add_combined_text_column,
    configure_logging,
    get_dataset_overview,
    get_descriptive_statistics,
    get_duplicate_row_count,
    get_duplicate_value_report,
    get_missing_value_report,
    get_null_percentage,
    get_text_quality_summary,
    load_excel_dataset,
)

configure_logging(logging.INFO)

DATASET_PATH = PROJECT_ROOT / "data" / "raw" / "Dataset.xlsx"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "exploration.csv"
REPORT_PATH = PROJECT_ROOT / "docs" / "EDA.md"

TEXT_COLUMNS = ["Title", "Opening Text", "Hit Sentence"]

## Load Dataset

In [ ]:
df = load_excel_dataset(DATASET_PATH)
overview = get_dataset_overview(df)

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(list(df.columns))
print("\nData types:")
print(df.dtypes)
print("\nSample rows:")
display(df.head())

## Missing Values and Duplicates

In [ ]:
missing_report = get_missing_value_report(df)
null_percentage = get_null_percentage(df)
duplicate_rows = get_duplicate_row_count(df)
duplicate_url_report = get_duplicate_value_report(df, "URL")
duplicate_title_report = get_duplicate_value_report(df, "Title")

print("Missing value report:")
display(missing_report)

print("Duplicate row count:", duplicate_rows)
print("Duplicate URL values:", len(duplicate_url_report))
display(duplicate_url_report)
print("Duplicate Title values:", len(duplicate_title_report))
display(duplicate_title_report)

print("Null percentage per column:")
display(null_percentage.to_frame("null_percentage"))

## Combined Text and Descriptive Statistics

In [ ]:
exploration_df = add_combined_text_column(df, TEXT_COLUMNS)
descriptive_stats = get_descriptive_statistics(exploration_df)
text_quality = get_text_quality_summary(exploration_df, TEXT_COLUMNS + ["combined_text"])

print("Descriptive statistics:")
display(descriptive_stats)

print("Text quality summary:")
display(text_quality)

print("Combined text samples:")
display(exploration_df[["Title", "Opening Text", "Hit Sentence", "combined_text"]].head())

## Save Phase 1 Outputs

In [ ]:
def dataframe_to_markdown(table: pd.DataFrame) -> str:
    """Render a compact DataFrame as a markdown table without extra dependencies."""
    table = table.copy()
    table.columns = [str(column) for column in table.columns]
    rows = [[str(value) for value in row] for row in table.to_numpy()]
    header = "| " + " | ".join(table.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(table.columns)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return "\n".join([header, separator, *body])


PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

exploration_df.to_csv(PROCESSED_PATH, index=False)

duplicate_url_rows = int(df["URL"].duplicated(keep=False).sum())
duplicate_title_rows = int(df["Title"].duplicated(keep=False).sum())
sentiment_counts = df["Sentiment"].value_counts(dropna=False).reset_index()
sentiment_counts.columns = ["sentiment", "count"]

observations = [
    "Driver and Sub driver are fully empty, which is expected before the classification phase but must be populated later.",
    "Hit Sentence has substantial missingness, so downstream text construction should safely fall back to Title and Opening Text.",
    "Reach is missing for a meaningful share of records, so dashboard reach aggregations will need explicit null handling.",
    "Date, Source Name, Title, Opening Text, and Hit Sentence contain missing values that should be reviewed before preprocessing.",
    "Duplicate URLs and duplicate titles indicate syndicated, repeated, or overlapping media mentions that require deduplication rules in the next phase.",
    "Sentiment labels are populated, but capitalization is inconsistent across values such as positive, Positive, neutral, and Negative.",
    "Text fields include snippets with ellipses and partial context, which may affect classification confidence if not handled carefully.",
]

recommendations = [
    "Define deterministic deduplication rules using URL, normalized title, source, and date before classification.",
    "Preserve the original Title, Opening Text, and Hit Sentence columns while creating derived text fields for modeling.",
    "Normalize whitespace and text encoding in a later preprocessing phase, but avoid changing raw evidence fields in Phase 1 outputs.",
    "Standardize categorical labels such as Sentiment only after documenting the raw distribution.",
    "Decide how missing Reach should be represented before building dashboards or weighted insights.",
    "Validate whether missing Date and Source Name records are acceptable or should be excluded as low-quality mentions.",
]

report = f"""# Exploratory Data Analysis Report

## Dataset Overview

- Source workbook: `data/raw/Dataset.xlsx`
- Sheet analyzed: first worksheet
- Number of records: {overview['rows']}
- Number of columns: {overview['columns']}
- Columns: {', '.join(overview['column_names'])}

## Missing Values

{dataframe_to_markdown(missing_report)}

## Duplicate Analysis

- Fully duplicated rows: {duplicate_rows}
- Rows belonging to duplicated URLs: {duplicate_url_rows}
- Unique duplicated URL values: {len(duplicate_url_report)}
- Rows belonging to duplicated titles: {duplicate_title_rows}
- Unique duplicated Title values: {len(duplicate_title_report)}

## Sentiment Label Profile

{dataframe_to_markdown(sentiment_counts)}

## Text Quality Summary

{dataframe_to_markdown(text_quality)}

## Data Quality Observations

{chr(10).join(f'- {item}' for item in observations)}

## Recommendations Before Preprocessing

{chr(10).join(f'- {item}' for item in recommendations)}

## Phase 1 Output

- Exploration dataset saved to `data/processed/exploration.csv`
- `combined_text` was created from Title, Opening Text, and Hit Sentence without cleaning or classification.
"""

REPORT_PATH.write_text(report, encoding="utf-8")

print(f"Saved exploration CSV to: {PROCESSED_PATH}")
print(f"Saved EDA report to: {REPORT_PATH}")